In [ ]:
import mlflow
from mlflow.models import infer_signature
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X,y = datasets.load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


params = {
    "solver": "lbfgs",
    "max_iter": 700,
    "random_state": 8888
} 

mlflow.set_tracking_uri(uri="https://boned-ankle-snooze.ngrok-free.dev") #link al servidor que se esta usando
mlflow.set_experiment("Ejemplo_2") #nombre del experimento

# toda la logica de entrenamiento y registro tiene que estar dentro de la function with mlflow.start_run
# para que los parametros y metricas se registren correctamente en MLflow
with mlflow.start_run():
    mlflow.log_params(params) #registro de parametros del modelo
    
    model = LogisticRegression()
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy) #registro de la metrica de precision del modelo
    
    signature = infer_signature(X_train, model.predict(X_train)) #inferencia de la firma del modelo
    
    # registro del modelo en MLflow con la firma y el ejemplo de entrada o input
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="logistic_reg_ejemplo2",
        signature=signature,
        input_example=X_train,
        registered_model_name="logistic_reg_ejemplo2"
    )
    
    mlflow.set_logged_model_tags(
        model_info.model_id,
        {"Training": "Iris dataset"}
    )
